# Notebook 3: Genetic Algorithm Optimization and Evaluation

GA optimizes model parameters using validation RMSE as fitness.

In [ ]:
import numpy as np
from deap import base, creator, tools, algorithms
from sklearn.metrics import mean_squared_error, mean_absolute_error


In [ ]:
# Fitness function: negative RMSE because GA maximizes fitness
def fitness_function(params):
    units=int(params[0])
    lr=params[1]
    model=build_lstm(units=units,lr=lr)
    X,y=create_dataset(imfs[0])
    X=X.reshape(X.shape[0],X.shape[1],1)
    model.fit(X,y,epochs=10,verbose=0)
    pred=model.predict(X,verbose=0)
    rmse=np.sqrt(mean_squared_error(y,pred))
    return (-rmse,)


In [ ]:
# GA population definition
creator.create('FitnessMax',base.Fitness,weights=(1.0,))
creator.create('Individual',list,fitness=creator.FitnessMax)
toolbox=base.Toolbox()
toolbox.register('individual',tools.initCycle,creator.Individual,
                (lambda:np.random.randint(32,128),lambda:np.random.uniform(0.0001,0.01)),n=1)
toolbox.register('population',tools.initRepeat,list,toolbox.individual)
toolbox.register('evaluate',fitness_function)
toolbox.register('mate',tools.cxTwoPoint)
toolbox.register('mutate',tools.mutGaussian,mu=0,sigma=0.1,indpb=0.2)
toolbox.register('select',tools.selTournament,tournsize=3)


In [ ]:
# Run Genetic Algorithm
population=toolbox.population(n=10)
algorithms.eaSimple(population,toolbox,cxpb=0.5,mutpb=0.2,ngen=20,verbose=True)
best=tools.selBest(population,1)[0]
print('Best parameters:',best)


In [ ]:
# Evaluation metrics
def evaluate(actual,predicted):
    rmse=np.sqrt(mean_squared_error(actual,predicted))
    mae=mean_absolute_error(actual,predicted)
    mape=np.mean(np.abs((actual-predicted)/actual))*100
    return {'RMSE':rmse,'MAE':mae,'MAPE':mape}
